# Vector Search Setup: Querido Diário RAG

## Purpose
Set up Databricks Vector Search for semantic retrieval over chunked gazette data.

## Components
1. **Vector Search Endpoint**: Compute resource for embeddings and similarity search
2. **Vector Search Index**: Delta Sync index that automatically embeds and indexes the gold layer chunks

## Configuration
- **Source Table**: `workspace.tcc_rag.gold_querido_diario_chunks` (from Phase 1)
- **Embedding Model**: `databricks-bge-large-en` (1024 dimensions, multilingual)
- **Primary Key**: `chunk_id`
- **Embedding Column**: `chunk_text`

## Pipeline Type Options
- **TRIGGERED**: Manual sync (on-demand)
- **CONTINUOUS**: Auto-sync when source table changes

For this initial setup, we'll use **TRIGGERED** for more control.

In [0]:
# Configuration
from databricks.sdk import WorkspaceClient
import pyspark.sql.functions as F

# TEST MODE: Set to True to test with 10 records first
TEST_MODE = False
TEST_RECORDS = 1000

# Vector Search configuration
ENDPOINT_NAME = "querido_diario_endpoint"
SOURCE_TABLE_FULL = "workspace.tcc_rag.gold_quality_variants_chunks"  # Full table (85k chunks)
SOURCE_TABLE_TEST = "workspace.tcc_rag.gold_quality_variants_chunks_test"  # Test table (10 chunks)
INDEX_NAME_BASE = "workspace.tcc_rag.querido_diario_vector_index"

PRIMARY_KEY = "chunk_id"
EMBEDDING_SOURCE_COLUMN = "chunk_text"  # Vector Search will embed this column
EMBEDDING_MODEL = "databricks-bge-large-en"  # Multilingual, 1024 dimensions

# Select configuration based on test mode
if TEST_MODE:
    SOURCE_TABLE = SOURCE_TABLE_TEST
    INDEX_NAME = f"{INDEX_NAME_BASE}_test"
    print(f"⚠️  TEST MODE ENABLED")
    print(f"   Using {TEST_RECORDS} records from test table")
else:
    SOURCE_TABLE = SOURCE_TABLE_FULL
    INDEX_NAME = INDEX_NAME_BASE
    print(f"🚀 FULL MODE")
    print(f"   Using full table with ~85k records")

# Initialize client
w = WorkspaceClient()

print(f"\n✓ Configuration loaded")
print(f"  Endpoint: {ENDPOINT_NAME}")
print(f"  Source: {SOURCE_TABLE}")
print(f"  Index: {INDEX_NAME}")
print(f"  Embedding Model: {EMBEDDING_MODEL}")
print(f"\n💡 Vector Search will embed '{EMBEDDING_SOURCE_COLUMN}' during indexing")

🚀 FULL MODE
   Using full table with ~85k records

✓ Configuration loaded
  Endpoint: querido_diario_endpoint
  Source: workspace.tcc_rag.gold_quality_variants_chunks
  Index: workspace.tcc_rag.querido_diario_vector_index
  Embedding Model: databricks-bge-large-en

💡 Vector Search will embed 'chunk_text' during indexing


In [0]:
# Create test table with 1000 records for initial testing
# This table contains chunk_text (no pre-computed embeddings)
# Vector Search will embed chunk_text during indexing

if TEST_MODE:
    print(f"Creating test table: {SOURCE_TABLE_TEST}")
    print(f"  Source: {SOURCE_TABLE_FULL}")
    print(f"  Records: {TEST_RECORDS}\n")
    
    # Select 1000 records from the full chunks table
    test_df = (
        spark.table(SOURCE_TABLE_FULL)
        .select(
            "chunk_id",
            "gazette_id",
            "quality_variant",
            "chunk_text",
            "chunk_index",
            "chunk_token_count"
        )
        .limit(TEST_RECORDS)
    )
    
    # Write to test table
    test_df.write.format("delta").mode("overwrite").saveAsTable(SOURCE_TABLE_TEST)
    
    # Verify
    test_count = spark.table(SOURCE_TABLE_TEST).count()
    print(f"✓ Test table created: {test_count} records")
    
    # Show sample
    print(f"\nSample records:")
    spark.table(SOURCE_TABLE_TEST).select(
        "chunk_id",
        "quality_variant",
        F.substring("chunk_text", 1, 50).alias("chunk_preview")
    ).show(5, truncate=False)
    
    print(f"\n💡 Vector Search will embed the 'chunk_text' column during indexing")
    print(f"   No pre-computed embeddings needed!")
else:
    print(f"✓ FULL MODE - will use existing table: {SOURCE_TABLE_FULL}")
    count = spark.table(SOURCE_TABLE_FULL).count()
    print(f"  Records: {count:,}")

✓ FULL MODE - will use existing table: workspace.tcc_rag.gold_quality_variants_chunks
  Records: 85,334


In [0]:
# Step 1: Create Vector Search Endpoint

from databricks.sdk.errors import NotFound

# Check if endpoint already exists
try:
    endpoint = w.vector_search_endpoints.get_endpoint(ENDPOINT_NAME)
    print(f"✓ Endpoint '{ENDPOINT_NAME}' already exists")
    print(f"  Status: {endpoint.endpoint_status.state.value if endpoint.endpoint_status else 'UNKNOWN'}")
except NotFound:
    print(f"Creating endpoint '{ENDPOINT_NAME}'...")
    print("⏳ This may take 5-10 minutes...")
    
    from databricks.sdk.service.vectorsearch import EndpointType
    wait = w.vector_search_endpoints.create_endpoint(
        name=ENDPOINT_NAME,
        endpoint_type=EndpointType.STANDARD
    )
    endpoint = wait.result()
    
    print(f"✓ Endpoint '{ENDPOINT_NAME}' created successfully")
    print(f"  Status: {endpoint.endpoint_status.state.value if endpoint.endpoint_status else 'UNKNOWN'}")

# Wait for endpoint to be online
import time
max_wait = 600  # 10 minutes
start_time = time.time()

while True:
    endpoint_info = w.vector_search_endpoints.get_endpoint(ENDPOINT_NAME)
    status = endpoint_info.endpoint_status.state.value if endpoint_info.endpoint_status else 'UNKNOWN'
    
    if status == 'ONLINE':
        print(f"\n✓ Endpoint is ONLINE and ready")
        break
    elif status in ['OFFLINE', 'PROVISIONING_FAILURE']:
        print(f"\n❌ Endpoint failed to provision. Status: {status}")
        raise Exception(f"Endpoint provisioning failed: {status}")
    else:
        elapsed = int(time.time() - start_time)
        print(f"  Status: {status} (elapsed: {elapsed}s)", end='\r')
        
        if elapsed > max_wait:
            print(f"\n⚠️ Timeout waiting for endpoint (>{max_wait}s)")
            break
        
        time.sleep(10)

✓ Endpoint 'querido_diario_endpoint' already exists
  Status: ONLINE

✓ Endpoint is ONLINE and ready


In [0]:
# Step 2: Create Vector Search Index

from databricks.sdk.errors import NotFound
from databricks.sdk.service.vectorsearch import (
    VectorIndexType, 
    DeltaSyncVectorIndexSpecRequest,
    EmbeddingSourceColumn,
    PipelineType
)

# Enable Change Data Feed on source table (required for Delta Sync indexes)
print(f"Enabling Change Data Feed on {SOURCE_TABLE}...")
spark.sql(f"ALTER TABLE {SOURCE_TABLE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
print(f"✓ Change Data Feed enabled\n")

# Check if index already exists
try:
    index_info = w.vector_search_indexes.get_index(INDEX_NAME)
    print(f"✓ Index '{INDEX_NAME}' already exists")
    status_msg = index_info.status.message if index_info.status else 'Unknown'
    is_ready = index_info.status.ready if index_info.status else False
    
    if is_ready:
        print(f"\n✅ Status: READY for queries")
        if index_info.status and hasattr(index_info.status, 'indexed_row_count'):
            print(f"   Indexed rows: {index_info.status.indexed_row_count:,}")
    else:
        print(f"\n⏳ Status: PROVISIONING")
        print(f"   {status_msg}")
        print("   Initial provisioning takes 15-30 minutes.")
        if index_info.status and hasattr(index_info.status, 'index_url'):
            print(f"   Monitor: {index_info.status.index_url}")
    
    print("\n📝 To recreate:")
    print(f"    w.vector_search_indexes.delete_index('{INDEX_NAME}')")
    print("    Then re-run this cell.")
    
except NotFound:
    print(f"Creating index '{INDEX_NAME}'...")
    print("⏳ Initial indexing may take several minutes...")
    
    delta_sync_spec = DeltaSyncVectorIndexSpecRequest(
        source_table=SOURCE_TABLE,
        pipeline_type=PipelineType.TRIGGERED,
        embedding_source_columns=[
            EmbeddingSourceColumn(
                name=EMBEDDING_SOURCE_COLUMN,
                embedding_model_endpoint_name=EMBEDDING_MODEL
            )
        ]
    )
    
    index = w.vector_search_indexes.create_index(
        name=INDEX_NAME,
        endpoint_name=ENDPOINT_NAME,
        primary_key=PRIMARY_KEY,
        index_type=VectorIndexType.DELTA_SYNC,
        delta_sync_index_spec=delta_sync_spec
    )
    
    print(f"✓ Index '{INDEX_NAME}' created successfully")
    print("\n📝 Note: Use TRIGGERED pipeline type for manual sync.")
    print("   To sync changes: w.vector_search_indexes.sync_index(INDEX_NAME)")
    print("\n⏳ Initial provisioning takes 15-30 minutes.")
    print("   The index is provisioning in the background.")
    if index.status and hasattr(index.status, 'index_url'):
        print(f"   Monitor progress: {index.status.index_url}")

Enabling Change Data Feed on workspace.tcc_rag.gold_quality_variants_chunks...
✓ Change Data Feed enabled

✓ Index 'workspace.tcc_rag.querido_diario_vector_index' already exists

⏳ Status: PROVISIONING
   Index is currently is in the process of syncing initial data. Check latest status: https://dbc-5ba61282-cdcc.cloud.databricks.com/explore/data/workspace/tcc_rag/querido_diario_vector_index
   Initial provisioning takes 15-30 minutes.
   Monitor: dbc-5ba61282-cdcc.cloud.databricks.com/api/2.0/vector-search/indexes/workspace.tcc_rag.querido_diario_vector_index

📝 To recreate:
    w.vector_search_indexes.delete_index('workspace.tcc_rag.querido_diario_vector_index')
    Then re-run this cell.


In [0]:
# Run this cell anytime to check if the index is ready

index_status = w.vector_search_indexes.get_index(INDEX_NAME)
is_ready = index_status.status.ready if index_status.status else False

if is_ready:
    print(f"✅ Index is READY for queries!")
    print(f"\n📊 Index Statistics:")
    if index_status.status and hasattr(index_status.status, 'indexed_row_count'):
        print(f"   Indexed rows: {index_status.status.indexed_row_count:,}")
    print(f"\n🎯 You can now run cell 5 to test semantic search.")
else:
    message = index_status.status.message if index_status.status else 'Unknown'
    print(f"⏳ Index is still provisioning...")
    print(f"\nStatus: {message}")
    print(f"\nThis typically takes 15-30 minutes.")
    if index_status.status and hasattr(index_status.status, 'index_url'):
        print(f"Monitor progress: {index_status.status.index_url}")

⏳ Index is still provisioning...

Status: Index is currently is in the process of syncing initial data. Check latest status: https://dbc-5ba61282-cdcc.cloud.databricks.com/explore/data/workspace/tcc_rag/querido_diario_vector_index

This typically takes 15-30 minutes.
Monitor progress: dbc-5ba61282-cdcc.cloud.databricks.com/api/2.0/vector-search/indexes/workspace.tcc_rag.querido_diario_vector_index


In [0]:
# Sync Vector Search Index with Latest Gold Table Data

print("🔄 Syncing vector search index with gold table...")
print(f"   Source: {SOURCE_TABLE}")
print(f"   Index: {INDEX_NAME}")
print("\n⏳ This may take several minutes depending on the number of new/updated rows...\n")

# Trigger sync
w.vector_search_indexes.sync_index(INDEX_NAME)

print("✓ Sync initiated successfully!")
print("\n📝 Note: The sync runs asynchronously in the background.")
print("   Run the 'Check Index Status' cell to monitor progress.")

🔄 Syncing vector search index with gold table...
   Source: workspace.tcc_rag.gold_quality_variants_chunks
   Index: workspace.tcc_rag.querido_diario_vector_index

⏳ This may take several minutes depending on the number of new/updated rows...

✓ Sync initiated successfully!

📝 Note: The sync runs asynchronously in the background.
   Run the 'Check Index Status' cell to monitor progress.


In [0]:
# Recreate Index with CONTINUOUS Mode
# CONTINUOUS mode automatically processes all rows and keeps index in sync

from databricks.sdk.service.vectorsearch import (
    VectorIndexType,
    DeltaSyncVectorIndexSpecRequest,
    EmbeddingSourceColumn,
    PipelineType
)

print("⚠️  IMPORTANT: This will delete and recreate the index.")
print("   Current index has 144 rows and pipeline stuck in CREATED state.")
print("   New index will use TRIGGERED mode with proper initialization.\n")

response = input("Proceed with deletion and recreation? (yes/no): ")

if response.lower() == "yes":
    # Check if index exists before trying to delete
    from databricks.sdk.errors import NotFound
    import time
    
    try:
        print("\n🔍 Checking if index exists...")
        w.vector_search_indexes.get_index(INDEX_NAME)
        print("   Found existing index")
        print("\n🗑️  Step 1: Deleting old index...")
        w.vector_search_indexes.delete_index(INDEX_NAME)
        print("✓ Old index deleted")
        print("\n⏳ Waiting 10 seconds for cleanup...")
        time.sleep(10)
    except NotFound:
        print("   Index already deleted (from previous attempt)")
        print("✓ Skipping deletion step")
    
    print("\n🆕 Step 2: Creating new index with TRIGGERED mode...")
    print("   (CONTINUOUS mode not supported in this workspace)\n")
    
    delta_sync_spec = DeltaSyncVectorIndexSpecRequest(
        source_table=SOURCE_TABLE,
        pipeline_type=PipelineType.TRIGGERED,  # Manual sync mode
        embedding_source_columns=[
            EmbeddingSourceColumn(
                name=EMBEDDING_SOURCE_COLUMN,
                embedding_model_endpoint_name=EMBEDDING_MODEL
            )
        ]
    )
    
    index = w.vector_search_indexes.create_index(
        name=INDEX_NAME,
        endpoint_name=ENDPOINT_NAME,
        primary_key=PRIMARY_KEY,
        index_type=VectorIndexType.DELTA_SYNC,
        delta_sync_index_spec=delta_sync_spec
    )
    
    print("✓ New index created with TRIGGERED mode!")
    print("\n📝 What happens now:")
    print("   • Pipeline will automatically run ONCE to process all 122,011 rows")
    print("   • This first run happens automatically (no sync needed)")
    print("   • After first run completes, future updates require manual sync")
    print("\n⏳ Initial indexing will take 20-40 minutes for 122K chunks.")
    print("   Monitor progress by re-running 'Check Index Status' cell.")
    print("\n🔄 To sync future updates after first run completes:")
    print("   w.vector_search_indexes.sync_index(INDEX_NAME)")
else:
    print("\n❌ Operation cancelled.")

⚠️  IMPORTANT: This will delete and recreate the index.
   Current index has 144 rows and pipeline stuck in CREATED state.
   New index will use TRIGGERED mode with proper initialization.



Proceed with deletion and recreation? (yes/no):  yes


🔍 Checking if index exists...
   Found existing index

🗑️  Step 1: Deleting old index...
✓ Old index deleted

⏳ Waiting 10 seconds for cleanup...

🆕 Step 2: Creating new index with TRIGGERED mode...
   (CONTINUOUS mode not supported in this workspace)

✓ New index created with TRIGGERED mode!

📝 What happens now:
   • Pipeline will automatically run ONCE to process all 122,011 rows
   • This first run happens automatically (no sync needed)
   • After first run completes, future updates require manual sync

⏳ Initial indexing will take 20-40 minutes for 122K chunks.
   Monitor progress by re-running 'Check Index Status' cell.

🔄 To sync future updates after first run completes:
   w.vector_search_indexes.sync_index(INDEX_NAME)


In [0]:
# Step 3: Test Semantic Search

# First verify the index exists and is ready
print(f"Checking index status...")
try:
    index_info = w.vector_search_indexes.get_index(INDEX_NAME)
    is_ready = index_info.status.ready if index_info.status else False
    
    if not is_ready:
        print(f"⚠️ Index is not ready yet. Status: {index_info.status.message if index_info.status else 'Unknown'}")
        print(f"   Please wait for indexing to complete, then re-run this cell.")
    else:
        print(f"✓ Index is ready\n")
        
        # Test query
        test_query = "decreto lei salário mínimo"
        
        print(f"🔍 Testing semantic search...")
        print(f"   Query: '{test_query}'\n")
        
        results = w.vector_search_indexes.query_index(
            index_name=INDEX_NAME,
            query_text=test_query,
            columns=["chunk_id", "chunk_text", "gazette_id", "quality_variant"],
            num_results=5
        )
        
        # Access result data
        result_data = results.result.data_array if results.result else []
        print(f"✓ Found {len(result_data)} results\n")
        print("=" * 80)
        
        # Display results
        for i, row in enumerate(result_data, 1):
            print(f"\nResult #{i}")
            print(f"  Chunk ID: {row[0]}")
            print(f"  Gazette ID: {row[2]}")
            print(f"  Quality Variant: {row[3]}")
            print(f"  Preview: {row[1][:200]}...")
            print("-" * 80)
            
except Exception as e:
    print(f"❌ Error: {e}")
    print(f"\n💡 Troubleshooting:")
    print(f"   1. Check if the index was deleted (Cell 8 deletes and recreates)")
    print(f"   2. Re-run Cell 5 to create the index")
    print(f"   3. Wait for indexing to complete (check Cell 6)")
    print(f"   4. Then re-run this cell")

Checking index status...
✓ Index is ready

🔍 Testing semantic search...
   Query: 'decreto lei salário mínimo'

✓ Found 5 results


Result #1
  Chunk ID: 99d999179cb6c52620e5a41ae7c017703ca179fa184d894386ae2ac49eac0ad7_baseline_chunk_35
  Gazette ID: 99d999179cb6c52620e5a41ae7c017703ca179fa184d894386ae2ac49eac0ad7
  Quality Variant: baseline
  Preview: ________________________________________
 Secretária MAÍRA RUFINO FISCHER

PORTARIA Nº 006, DE 31 DE JANEIRO DE 2025.
 A SECRETÁRIA DE ADMINISTRAÇÃO, no uso da delegação prevista no inciso III, art. 2...
--------------------------------------------------------------------------------

Result #2
  Chunk ID: 99d999179cb6c52620e5a41ae7c017703ca179fa184d894386ae2ac49eac0ad7_baseline_chunk_37
  Gazette ID: 99d999179cb6c52620e5a41ae7c017703ca179fa184d894386ae2ac49eac0ad7
  Quality Variant: baseline
  Preview: o subsistindo razão, portanto, para a 
modificação do Parecer nº 0934/2024, da Procuradoria Consultiva, aprovado pelas egrégias instânci

In [0]:
# Step 4: Test Filtered Search (Hybrid Search)

filtered_query = "licitação pública obras"

print(f"🔍 Testing filtered semantic search...")
print(f"   Query: '{filtered_query}'")

import json

filtered_results = w.vector_search_indexes.query_index(
    index_name=INDEX_NAME,
    query_text=filtered_query,
    columns=["chunk_id", "chunk_text", "gazette_id", "quality_variant"],
    num_results=3
)

print(f"✓ Found {len(filtered_results.result.data_array if filtered_results.result else [])} results\n")
print("=" * 80)

for i, row in enumerate(filtered_results.result.data_array if filtered_results.result else [], 1):
    print(f"\nResult #{i}")
    print(f"  Chunk ID: {row[0]}")
    print(f"  Gazette ID: {row[2]}")
    print(f"  Quality Variant: {row[3]}")
    print(f"  Preview: {row[1][:200]}...")
    print("-" * 80)

print("\n✓ Hybrid search (semantic + filters) working correctly!")

🔍 Testing filtered semantic search...
   Query: 'licitação pública obras'
✓ Found 3 results


Result #1
  Chunk ID: 99d999179cb6c52620e5a41ae7c017703ca179fa184d894386ae2ac49eac0ad7_baseline_chunk_14
  Gazette ID: 99d999179cb6c52620e5a41ae7c017703ca179fa184d894386ae2ac49eac0ad7
  Quality Variant: baseline
  Preview: e 27 de dezembro de 2024, que dispõe sobre a estrutura e funcionamento da Administração 
Direta e Indireta do Peode Executivo Municipal;

CONSIDERANDO a necessidade de reorganizar as estruturas de ges...
--------------------------------------------------------------------------------

Result #2
  Chunk ID: 99d999179cb6c52620e5a41ae7c017703ca179fa184d894386ae2ac49eac0ad7_baseline_chunk_10
  Gazette ID: 99d999179cb6c52620e5a41ae7c017703ca179fa184d894386ae2ac49eac0ad7
  Quality Variant: baseline
  Preview: ANDO a Lei Municipal nº 19.337, de 27 de dezembro de 2024, que dispõe sobre a estrutura e funcionamento da 
Administração Direta e Indireta do Poder Executivo Municipal;

CON

In [0]:
# RAG Example: Combining Vector Search + LLM
# This demonstrates a complete Retrieval Augmented Generation pipeline

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

# Step 1: Vector Search - Retrieve relevant chunks
user_question = "Quantas decretos tivemos por secretaria? Inclua uma tabela com os resultados e data do decreto"

print(f"🔍 User Question: {user_question}\n")
print("Step 1: Retrieving relevant chunks from vector index...")

# Query vector index
retrieval_results = w.vector_search_indexes.query_index(
    index_name=INDEX_NAME,
    query_text=user_question,
    columns=["chunk_text", "gazette_id", "quality_variant"],
    num_results=5  # Get top 5 most relevant chunks
)

result_data = retrieval_results.result.data_array if retrieval_results.result else []
print(f"✓ Retrieved {len(result_data)} relevant chunks\n")

# Step 2: Format context from retrieved chunks
context_parts = []
for i, row in enumerate(result_data, 1):
    chunk_text = row[0]
    gazette_id = row[1]
    quality_variant = row[2]
    context_parts.append(f"Documento {i} (Gazette: {gazette_id}, Variant: {quality_variant}):\n{chunk_text}")

context = "\n\n".join(context_parts)

print("Step 2: Building prompt with retrieved context...")
print(f"   Context length: {len(context)} characters\n")

# Step 3: LLM - Generate answer using retrieved context
print("Step 3: Generating answer with LLM...\n")

# Create prompt with retrieved context
system_prompt = """Você é um assistente especializado em legislação municipal brasileira.
Responda a pergunta do usuário usando APENAS as informações fornecidas no contexto.
Se a informação não estiver no contexto, diga que não encontrou essa informação específica.
Cite os documentos relevantes na sua resposta."""

user_prompt = f"""Contexto (documentos municipais de Recife):

{context}

---

Pergunta: {user_question}

Resposta:"""

# Call Databricks Foundation Model API
w_serving = WorkspaceClient()
response = w_serving.serving_endpoints.query(
    name="databricks-meta-llama-3-3-70b-instruct",  # Using Llama 3.3 70B
    messages=[
        ChatMessage(role=ChatMessageRole.SYSTEM, content=system_prompt),
        ChatMessage(role=ChatMessageRole.USER, content=user_prompt)
    ],
    max_tokens=500,
    temperature=0.1  # Lower temperature for factual answers
)

# Extract answer
answer = response.choices[0].message.content

print("=" * 80)
print("📝 ANSWER:")
print("=" * 80)
print(answer)
print("=" * 80)

print("\n✓ RAG pipeline complete!")
print("\n💡 This combines:")
print("   1. Vector search for semantic retrieval")
print("   2. LLM for natural language synthesis")
print("   3. Filtered search for targeted results (state_code = AL)")

🔍 User Question: Quantas decretos tivemos por secretaria? Inclua uma tabela com os resultados e data do decreto

Step 1: Retrieving relevant chunks from vector index...
✓ Retrieved 5 relevant chunks

Step 2: Building prompt with retrieved context...
   Context length: 15638 characters

Step 3: Generating answer with LLM...

📝 ANSWER:
De acordo com as informações fornecidas nos documentos, podemos identificar os seguintes decretos por secretaria:

| Secretaria | Número de Decretos | Data do Decreto |
| --- | --- | --- |
| Secretaria de Educação | 1 | 05 de fevereiro de 2025 |
| Secretaria de Finanças | 1 | 05 de fevereiro de 2025 |
| Secretaria de Administração | 1 | 31 de janeiro de 2025 |
| Secretaria de Desenvolvimento Urbano e Licenciamento | 1 | 05 de fevereiro de 2025 |
| Secretaria de Ordem Pública e Segurança | 1 | 05 de fevereiro de 2025 |

Observação: O Decreto nº 38.519, de 05 de fevereiro de 2025, menciona várias secretarias, mas parece ser um decreto único que afeta várias 

## 🎉 Vector Search Setup Complete!

Your Vector Search index is now ready for RAG applications.

### What You Can Do Now

1. **Query the index programmatically**
   ```python
   from databricks.vector_search.client import VectorSearchClient
   vsc = VectorSearchClient()
   index = vsc.get_index("workspace.tcc_rag.querido_diario_vector_index")
   
   results = index.similarity_search(
       query_text="seu texto de busca",
       num_results=5
   )
   ```

2. **Use with Databricks AI Playground**
   - Navigate to AI Playground
   - Select your Vector Search index as a retrieval source
   - Test RAG queries interactively

3. **Build a RAG Application**
   - Use the index in a Databricks App
   - Integrate with LangChain or LlamaIndex
   - Deploy as a Model Serving endpoint

### Index Management

**Manual Sync** (TRIGGERED mode):
```python
index.sync()
```

**Check Index Status**:
```python
vsc.get_index("workspace.tcc_rag.querido_diario_vector_index")
```

**Delete Index** (if needed):
```python
vsc.delete_index("workspace.tcc_rag.querido_diario_vector_index")
```

### Available Filter Dimensions

You can filter results by any of these metadata columns:
- `territory_name`: Municipality name
- `state_code`: State abbreviation (e.g., "AL", "SP")
- `publication_date`: Date filter (e.g., `{">" :"2024-01-01"}`)
- `territory_id`: IBGE territory code

---

**📚 Next Steps**: Build your RAG application or explore the Playground!

In [0]:
# RAG Example: Combining Vector Search + LLM
# This demonstrates a complete Retrieval Augmented Generation pipeline

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

# Step 1: Vector Search - Retrieve relevant chunks
user_question = "Quais são as regras sobre aposentadoria em Alagoas?"

print(f"🔍 User Question: {user_question}\n")
print("Step 1: Retrieving relevant chunks from vector index...")

# Query vector index
retrieval_results = w.vector_search_indexes.query_index(
    index_name=INDEX_NAME,
    query_text=user_question,
    columns=["chunk_text", "territory_name", "state_code", "publication_date"],
    filters_json=json.dumps({"state_code": "PE"}),  # Filter to Alagoas
    num_results=5  # Get top 5 most relevant chunks
)

result_data = retrieval_results.result.data_array if retrieval_results.result else []
print(f"✓ Retrieved {len(result_data)} relevant chunks\n")

# Step 2: Format context from retrieved chunks
context_parts = []
for i, row in enumerate(result_data, 1):
    chunk_text = row[0]
    territory = row[1]
    date = row[3]
    context_parts.append(f"Documento {i} ({territory}, {date}):\n{chunk_text}")

context = "\n\n".join(context_parts)

print("Step 2: Building prompt with retrieved context...")
print(f"   Context length: {len(context)} characters\n")

# Step 3: LLM - Generate answer using retrieved context
print("Step 3: Generating answer with LLM...\n")

# Create prompt with retrieved context
system_prompt = """Você é um assistente especializado em legislação municipal brasileira.
Responda a pergunta do usuário usando APENAS as informações fornecidas no contexto.
Se a informação não estiver no contexto, diga que não encontrou essa informação específica.
Cite os documentos relevantes na sua resposta."""

user_prompt = f"""Contexto (documentos municipais de Alagoas):

{context}

---

Pergunta: {user_question}

Resposta:"""

# Call Databricks Foundation Model API
w_serving = WorkspaceClient()
response = w_serving.serving_endpoints.query(
    name="databricks-meta-llama-3-1-70b-instruct",  # Using Llama 3.1 70B
    messages=[
        ChatMessage(role=ChatMessageRole.SYSTEM, content=system_prompt),
        ChatMessage(role=ChatMessageRole.USER, content=user_prompt)
    ],
    max_tokens=500,
    temperature=0.1  # Lower temperature for factual answers
)

# Extract answer
answer = response.choices[0].message.content

print("=" * 80)
print("📝 ANSWER:")
print("=" * 80)
print(answer)
print("=" * 80)

print("\n✓ RAG pipeline complete!")
print("\n💡 This combines:")
print("   1. Vector search for semantic retrieval")
print("   2. LLM for natural language synthesis")
print("   3. Filtered search for targeted results (state_code = AL)")